# GSM Thermodynamic Box Framework

This notebook demonstrates the **GSMThermodynBox** class, which provides the foundational thermodynamic framework for the GSM (Generalized Standard Materials) modeling approach.

## Architecture Overview

The new three-layer architecture separates concerns into:

1. **GSMThermodynBox** - Static thermodynamic framework (this notebook)
2. **GSMStateFn** - Dynamic state function accessor (cursor)  
3. **StateFunctionTag** - Clear enum for state function identification

## Key Concepts

- **Thermodynamic Box**: Defines the static structure with four corners (U, F, H, G)
- **Corner Variables**: T, S (always present) + configurable eps_vars, sig_vars
- **Internal Variables**: Eps_vars (extensive), Sig_vars (intensive forces)
- **Natural Variables**: Each state function has specific "natural" variable combinations
- **Legendre Transformations**: Mathematical relationships between state functions

This notebook focuses on the **configuration and framework** aspects.

In [1]:
# Import required modules
import sympy as sp
import numpy as np
from gsm_thermodyn_box import GSMThermodynBox, StateFunctionTag

# Configure SymPy for better display
sp.init_printing()

print("GSM Thermodynamic Box Framework - Demo 01")
print("=" * 45)

GSM Thermodynamic Box Framework - Demo 01


## 1. Basic Thermodynamic Box Configuration

Let's start by creating a simple thermodynamic box with basic mechanical variables.

In [10]:
# Setup and imports
import sys
import os
import importlib

# Add the core2 module to the path
current_dir = os.path.dirname(os.path.abspath('__file__' if '__file__' in globals() else 'gsm_thermodyn_box_01.ipynb'))
sys.path.insert(0, current_dir)

# Force reload modules to get latest changes
if 'gsm_thermodyn_box' in sys.modules:
    importlib.reload(sys.modules['gsm_thermodyn_box'])
if 'gsm_state_fn' in sys.modules:
    importlib.reload(sys.modules['gsm_state_fn'])

from gsm_thermodyn_box import GSMThermodynBox, StateFunctionTag
import sympy as sp

# Create the thermodynamic box
box = GSMThermodynBox()

print("Thermodynamic box created successfully!")
print(f"Available state functions: {[tag.name for tag in StateFunctionTag]}")

# Validate the configuration
if box.validate_configuration():
    print("\n✓ Configuration is thermodynamically valid")

Thermodynamic box created successfully!
Available state functions: ['INTERNAL_ENERGY', 'HELMHOLTZ', 'ENTHALPY', 'GIBBS']

✓ Configuration is thermodynamically valid


In [3]:
# Display comprehensive box overview
box.print_box_overview()

GSM Thermodynamic Box Configuration

Corner Variables:
  Temperature (T): T
  Entropy (S): S
  External eps: (eps,)
  External sig: (sig,)

Internal Variables:
  Internal Eps: (Eps,)
  Internal Sig: (Sig,)

Material Parameters:
  Parameters: (E, omega)

State Function Templates:
  U: U(S, eps, Eps)
    Natural: [S, eps, Eps]
    Conjugate: [T, sig, Sig]
  F: F(T, eps, Eps)
    Natural: [T, eps, Eps]
    Conjugate: [S, sig, Sig]
  H: H(S, sig, Eps)
    Natural: [S, sig, Eps]
    Conjugate: [T, eps, Sig]
  G: G(T, sig, Eps)
    Natural: [T, sig, Eps]
    Conjugate: [S, eps, Sig]



## 2. Natural Variables and Thermodynamic Relationships

The core concept in thermodynamics is that each state function has "natural variables" - the specific combination of extensive and intensive variables that make the function well-defined and convex.

In [4]:
# Display the natural variables mapping
box.print_natural_variables_table()

Natural Variables Mapping
Each state function has natural (independent) and conjugate (derivative) variables

Function     Natural Variables         Conjugate Variables      
--------------------------------------------------------------
U(...)       S, eps, Eps               T, sig, Sig              
F(...)       T, eps, Eps               S, sig, Sig              
H(...)       S, sig, Eps               T, eps, Sig              
G(...)       T, sig, Eps               S, eps, Sig              

Legend:
  T: Temperature, S: Entropy
  eps: External strain, sig: External stress
  Eps: Internal strain, Sig: Internal stress


In [ ]:
# Examine natural variables for each state function
print("Detailed Natural Variables Analysis:")
print("=" * 40)

for state_fn_tag in StateFunctionTag:
    natural_vars, conjugate_vars = box.get_natural_variables(state_fn_tag)
    
    print(f"\n{state_fn_tag.value} State Function:")
    print(f"  Template: {box.state_function_templates[state_fn_tag]}")
    print(f"  Natural (independent): {[str(v) for v in natural_vars]}")
    print(f"  Conjugate (derivatives): {[str(v) for v in conjugate_vars]}")
    
    # Physical interpretation
    if state_fn_tag == StateFunctionTag.HELMHOLTZ:
        print("  → Ideal for displacement-controlled experiments")
    elif state_fn_tag == StateFunctionTag.GIBBS:
        print("  → Ideal for load-controlled experiments") 
    elif state_fn_tag == StateFunctionTag.INTERNAL_ENERGY:
        print("  → Ideal for isolated systems")
    elif state_fn_tag == StateFunctionTag.ENTHALPY:
        print("  → Ideal for isentropic processes under load")

## 3. Legendre Transformation Framework

The thermodynamic box provides the complete mapping of Legendre transformations between all state functions. These transformations allow converting between different thermodynamic perspectives.

In [ ]:
# Display the complete transformation mapping
box.print_transformation_table()

In [ ]:
# Analyze the transformation graph structure
print("Thermodynamic Square - Adjacency Structure:")
print("=" * 45)

transformation_graph = box.get_transformation_graph()

for state_fn, neighbors in transformation_graph.items():
    neighbor_names = [fn.value for fn in neighbors]
    print(f"{state_fn.value} can directly transform to: {neighbor_names}")

print()
print("Key Insights:")
print("• Each state function connects to exactly 2 neighbors (thermodynamic square)")
print("• Direct transformations change exactly one variable pair (T↔S or ε↔σ)")
print("• Diagonal transformations change both pairs simultaneously")
print("• All transformations are reversible with opposite coefficients")

## 4. Creating State Function Accessors

The thermodynamic box serves as a factory for creating state function accessors (GSMStateFn objects) that handle the computational aspects.

In [12]:
# Create a state function accessor
state_fn = box.create_state_fn()
print(f"Created state function accessor: {type(state_fn)}")
print(f"Initial state function: {state_fn.current_state_fn_tag.name}")

# The accessor provides methods for transformations and calculations
print(f"\nAvailable methods:")
methods = [method for method in dir(state_fn) if not method.startswith('_')]
for method in sorted(methods[:10]):  # Show first 10
    print(f"  - {method}")
    
print("  ... and more")

# Show the cursor pattern
print(f"\nCursor pattern demonstration:")
print(f"Box defines the playground: {len(box.NATURAL_VARIABLES_MAPPING)} state functions")
print(f"Accessor provides the cursor: currently at '{state_fn.current_state_fn_tag.name}'")

Created state function accessor: <class 'gsm_state_fn.GSMStateFn'>
Initial state function: INTERNAL_ENERGY

Available methods:
  - F
  - G
  - H
  - U
  - compute_constitutive_relation
  - compute_constitutive_relations
  - current_state_fn_tag
  - expressions
  - get_all_expressions
  - get_current_expression
  ... and more

Cursor pattern demonstration:
Box defines the playground: 4 state functions
Accessor provides the cursor: currently at 'INTERNAL_ENERGY'


## 5. Conjugate Pairs and Thermodynamic Relationships

The framework automatically manages the conjugate pairs between state functions and their natural variables.

In [14]:
# Examine conjugate pairs for each state function
print("Conjugate pairs in the thermodynamic framework:")
print("=" * 50)

for state_fn_tag in StateFunctionTag:
    natural_vars, conjugate_vars = box.get_natural_variables(state_fn_tag)
    print(f"\n{state_fn_tag.name} ({state_fn_tag.value}):")
    print(f"  Natural variables: {[str(var) for var in natural_vars]}")
    print(f"  Conjugate variables: {[str(var) for var in conjugate_vars]}")
    
    # Show the thermodynamic relationship
    print(f"  Relationship: d{state_fn_tag.value} = ", end="")
    terms = []
    for nat_var, conj_var in zip(natural_vars, conjugate_vars):
        terms.append(f"{conj_var} d{nat_var}")
    print(" + ".join(terms))

# Show transformation relationships
print(f"\n\nTransformation examples:")
print("=" * 30)
transformations = list(box.TRANSFORMATION_MAPPING.keys())[:3]  # Show first 3
for transformation in transformations:
    operation = box.TRANSFORMATION_MAPPING[transformation]
    print(f"{transformation[0].name} → {transformation[1].name}: {operation}")

Conjugate pairs in the thermodynamic framework:

INTERNAL_ENERGY (U):
  Natural variables: ['S']
  Conjugate variables: ['T']
  Relationship: dU = T dS

HELMHOLTZ (F):
  Natural variables: ['T']
  Conjugate variables: ['S']
  Relationship: dF = S dT

ENTHALPY (H):
  Natural variables: ['S']
  Conjugate variables: ['T']
  Relationship: dH = T dS

GIBBS (G):
  Natural variables: ['T']
  Conjugate variables: ['S']
  Relationship: dG = S dT


Transformation examples:
INTERNAL_ENERGY → HELMHOLTZ: (-1, 0)
HELMHOLTZ → INTERNAL_ENERGY: (1, 0)
INTERNAL_ENERGY → ENTHALPY: (0, 1)


## 6. Validation and Consistency

The framework includes built-in validation to ensure thermodynamic consistency.

In [16]:
# Test validation features
print("Framework validation:")
print("=" * 25)

# Validate the framework structure
try:
    is_valid = box.validate_configuration()
    print(f"Framework validation: {'PASSED' if is_valid else 'FAILED'}")
except Exception as e:
    print(f"Validation error: {e}")

# Show variable consistency
print(f"\nVariable consistency:")
all_vars = set()
for state_fn_tag in StateFunctionTag:
    natural_vars, conjugate_vars = box.get_natural_variables(state_fn_tag)
    all_vars.update(str(var) for var in natural_vars)
    all_vars.update(str(var) for var in conjugate_vars)

print(f"Total unique variables across all state functions: {len(all_vars)}")
print(f"Variables: {sorted(all_vars)}")

# Check transformation completeness
print(f"\nTransformation completeness:")
expected_transformations = len(StateFunctionTag) * (len(StateFunctionTag) - 1)
actual_transformations = len(box.TRANSFORMATION_MAPPING)
print(f"Expected transformations: {expected_transformations}")
print(f"Actual transformations: {actual_transformations}")
print(f"Coverage: {actual_transformations/expected_transformations*100:.1f}%")

Framework validation:
Framework validation: PASSED

Variable consistency:
Total unique variables across all state functions: 2
Variables: ['S', 'T']

Transformation completeness:
Expected transformations: 12
Actual transformations: 12
Coverage: 100.0%


## 7. Summary

The `GSMThermodynBox` provides a clean, static framework for thermodynamic state function management with:

- **Static Configuration**: All mappings and relationships are class-level constants
- **Validation**: Built-in consistency checking for the thermodynamic framework
- **Factory Pattern**: Creates `GSMStateFn` accessor objects for computational work
- **Separation of Concerns**: Framework definition is separate from computational operations

### Key Benefits:

1. **Performance**: Class-level constants eliminate method call overhead
2. **Clarity**: Clear separation between static framework and dynamic computation
3. **Maintainability**: Centralized configuration with validation
4. **Extensibility**: Easy to add new state functions or modify relationships

### Next Steps:

- See `gsm_state_fn_01.ipynb` for examples of using the state function accessors
- The cursor pattern allows efficient navigation between different thermodynamic potentials
- Integration with existing `GSMSymbBox` remains compatible